In [ ]:
# Required Imports
import os
import sys
import numpy as np
import tensorflow as tf
from keras.models import load_model
from keras.preprocessing.text import tokenizer_from_json
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import json
from tqdm import tqdm
import logging
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

In [ ]:
# Suppress TensorFlow warnings and logs
tf.get_logger().setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

print("Starting BLEU Score Analysis Script...")

Starting BLEU Score Analysis Script...


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive Mounted Successfully.")

sys.path.insert(0, '/content/drive/MyDrive/CSCK507_EMA_GroupC_Final')
import config

model_file = os.path.join(config.MAIN_DIR_PATH, config.SEQ2SEQ_WA_MODEL)
tokenizer_file = os.path.join(config.MAIN_DIR_PATH, config.TOKENIZER_FILE)
test_data_file = os.path.join(config.MAIN_DIR_PATH, config.TEST_DATA)

print(f"Model File Path: {model_file}")
print(f"Tokenizer File Path: {tokenizer_file}")
print(f"Test Data File Path: {test_data_file}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive Mounted Successfully.
Model File Path: /content/drive/MyDrive/CSCK507_EMA_GroupC_Final/seq2seq_with_attention_model_RMSPROP.h5
Tokenizer File Path: /content/drive/MyDrive/CSCK507_EMA_GroupC_Final/tokenizer.json
Test Data File Path: /content/drive/MyDrive/CSCK507_EMA_GroupC_Final/test_data.npz


In [ ]:
# Load the Pre-trained Model
seq2seq_model = load_model(model_file)
print("Model loaded successfully!")

# Print the model summary to verify its structure
seq2seq_model.summary()

Model loaded successfully!
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 Encoder_Input (InputLayer)  [(None, 30)]                 0         []                            
                                                                                                  
 Decoder_Input (InputLayer)  [(None, 30)]                 0         []                            
                                                                                                  
 encoder_embedding (Embeddi  (None, 30, 96)               6149049   ['Encoder_Input[0][0]']       
 ng)                                                      6                                       
                                                                                                  
 Decoder_Embedding (Embeddi  (None, 30, 96)               6149049  

In [ ]:
# Load the tokenizer
with open(tokenizer_file, 'r') as f:
    tokenizer_data = json.load(f)
    tokenizer = tf.keras.preprocessing.text.tokenizer_from_json(tokenizer_data)

# Create index mappings using the correct word-level index
target_token_index = tokenizer.word_index
reverse_target_word_index = {i: word for word, i in target_token_index.items()}  # Ensure word-level mapping

print(f"Tokenizer Loaded. Number of Tokens: {len(target_token_index)}")

Tokenizer Loaded. Number of Tokens: 640525


In [ ]:
# Load the Test Data
test_data = np.load(test_data_file)
X_test_enc, X_test_dec, y_test = test_data['X_test_enc'], test_data['X_test_dec'], test_data['y_test']
print(f"Test Data Loaded. Number of Test Samples: {X_test_enc.shape[0]}")

Test Data Loaded. Number of Test Samples: 2025386


## **Model Extraction: Seq2Seq Model with Attention**

In [ ]:
# Extract the encoder
encoder_inputs = seq2seq_model.input[0]  # Encoder input is the first input
encoder_outputs, encoder_state = seq2seq_model.get_layer('Encoder_GRU').output

encoder_model = tf.keras.Model(encoder_inputs, [encoder_outputs, encoder_state])

# Extract the decoder
decoder_inputs = seq2seq_model.input[1]  # Decoder input is the second input
decoder_state_input_h = tf.keras.Input(shape=(64,), dtype='bfloat16', name='decoder_hidden_state')
encoder_outputs_input = tf.keras.Input(shape=(None, 64), dtype='bfloat16', name='encoder_outputs')

# Extract the embedding and GRU layers
decoder_embedding_layer = seq2seq_model.get_layer('Decoder_Embedding')
decoder_gru_layer = seq2seq_model.get_layer('Decoder_GRU')
attention_layer = seq2seq_model.get_layer('Attention_Concatenate')
dense_layer = seq2seq_model.get_layer('Decoder_Dense')

# Decoder embedding for the current input word
decoder_embedding_outputs = decoder_embedding_layer(decoder_inputs)

# Decoder GRU processing one time step at a time
decoder_gru_output, decoder_state_h = decoder_gru_layer(decoder_embedding_outputs, initial_state=[decoder_state_input_h])

# Compute the attention context vector
score = tf.matmul(decoder_gru_output, encoder_outputs_input, transpose_b=True)
attention_weights = tf.nn.softmax(score, axis=-1)
context_vector = tf.matmul(attention_weights, encoder_outputs_input)

# Concatenate the context vector with the decoder's GRU output
decoder_context_concat = tf.keras.layers.Concatenate(axis=-1)([decoder_gru_output, context_vector])

# Pass through the dense layer
decoder_outputs = dense_layer(decoder_context_concat)

# Define the decoder model for inference
decoder_model = tf.keras.Model(
    [decoder_inputs, decoder_state_input_h, encoder_outputs_input],
    [decoder_outputs, decoder_state_h, context_vector]
)

## **Greedy Decoding**

In [ ]:
def batch_greedy_decode(input_seqs, encoder_model, decoder_model, target_token_index, reverse_target_word_index, max_decoder_seq_length):
    """
    Perform Greedy Search to decode a batch of sequences using the encoder and decoder models with attention.
    """
    batch_size = input_seqs.shape[0]

    # Encode the input sequences to get the encoder outputs and the initial state
    encoder_outputs, encoder_state = encoder_model.predict(input_seqs, verbose=0)

    # Initialize the target sequences with the <SOS> token
    start_token_index = target_token_index.get('<SOS>', 0)
    target_seqs = np.full((batch_size, 1), start_token_index, dtype='int32')

    decoded_sentences = [[] for _ in range(batch_size)]
    finished = [False] * batch_size
    decoder_state = encoder_state  # Initialize decoder state with encoder state

    for _ in range(max_decoder_seq_length):
        # Use the decoder to predict the next tokens, incorporating attention
        decoder_outputs, decoder_state, attention_context_vector = decoder_model.predict(
            [target_seqs, decoder_state, encoder_outputs],
            verbose=0
        )

        # Select the token with the highest probability
        sampled_token_indices = np.argmax(decoder_outputs[:, -1, :], axis=1)

        for i, sampled_token_index in enumerate(sampled_token_indices):
            if not finished[i]:
                sampled_word = reverse_target_word_index.get(sampled_token_index, '')

                if sampled_word == '<EOS>':
                    finished[i] = True
                else:
                    decoded_sentences[i].append(sampled_word)

        # Update the target sequences for the next iteration
        target_seqs = sampled_token_indices[:, np.newaxis]

    return [' '.join(sentence) for sentence in decoded_sentences]

def decode_indices_to_words(sequence, reverse_target_word_index):
    """
    Convert a sequence of indices to a readable string using the reverse target word index.
    """
    words = []
    for index in sequence:
        word = reverse_target_word_index.get(index, '')
        if word == '<EOS>':
            break
        words.append(word)

    return ' '.join(words).strip()

## **BLEU Score Evaluation**

In [ ]:
# BLEU Score Evaluation
subset_size = 2000
batch_size = 10
bleu_scores = []
running_sum = 0

# Loop through the test dataset in batches
for i in tqdm(range(0, subset_size, batch_size), desc="Calculating BLEU Scores"):
    batch_end = min(i + batch_size, subset_size)

    # Get the input sequences for this batch
    input_seqs = X_test_enc[i:batch_end]

    # Perform greedy decoding to generate candidate translations
    candidate_sentences = batch_greedy_decode(
        input_seqs,
        encoder_model,
        decoder_model,
        target_token_index,
        reverse_target_word_index,
        max_decoder_seq_length=30
    )

    # Calculate BLEU score for each sentence in the batch
    for j, candidate_sentence in enumerate(candidate_sentences):
        reference_sentence = decode_indices_to_words(y_test[i + j], reverse_target_word_index)
        reference = [reference_sentence.split()]
        candidate = candidate_sentence.split()

        # Calculate BLEU score for the reference and candidate sentence
        bleu_score = sentence_bleu(reference, candidate, smoothing_function=SmoothingFunction().method1)
        bleu_scores.append(bleu_score)
        running_sum += bleu_score

    # Print real-time average BLEU score every 5 batches
    if (i // batch_size) % 5 == 0:
        print(f"Real-time Average BLEU Score after {i + batch_size} samples: {running_sum / len(bleu_scores):.4f}")

# Final average BLEU score across all batches
average_bleu_score = np.mean(bleu_scores)
print(f"\nFinal Average BLEU Score for the Subset of Test Data: {average_bleu_score:.4f}")

Calculating BLEU Scores:   0%|          | 1/200 [00:10<33:22, 10.06s/it]

Real-time Average BLEU Score after 10 samples: 0.0062


Calculating BLEU Scores:   3%|▎         | 6/200 [01:01<33:19, 10.31s/it]

Real-time Average BLEU Score after 60 samples: 0.0094


Calculating BLEU Scores:   6%|▌         | 11/200 [01:52<32:08, 10.20s/it]

...

Real-time Average BLEU Score after 1910 samples: 0.0111


Calculating BLEU Scores:  98%|█████████▊| 196/200 [33:40<00:41, 10.33s/it]

Real-time Average BLEU Score after 1960 samples: 0.0111


Calculating BLEU Scores: 100%|██████████| 200/200 [34:21<00:00, 10.31s/it]


Final Average BLEU Score for the Subset of Test Data: 0.0110
